# 15 · IGC atmospheric truth (high-precision sensor dataset)

Builds a calibrated, multi-variable atmospheric ground-truth benchmark (`database/truth/`) directly from the cleaned 5 s competition fixes (`database/fixes/`), without re-parsing the 153k raw IGC zip archives.

### What it improves over earlier tables

1. **Dual-DEM altitude audit (`truth_flights`):** filters out the ~24% of tracks without landing calibration and low-rate live-trackers, and cross-checks `takeoff_agl_raw − landing_agl_raw` against the Copernicus GLO-30 DEM to separate `gold` (`step <= 2 s`, drift `<= 40 m`) and `silver` flights.
2. **Phase-free thermal wind (`truth_thermals`, `truth_wind_groups`):** replaces endpoint `(end − start) / dt` drift with two independent kinematics estimators — a **Huber-weighted velocity hodograph circle fit** and **air-relative integer-turn displacement** (both exact at 5 s and valid even when wind exceeds glider airspeed) — gated by octant coverage, fitted circling airspeed, radial residual, and method agreement, with vertical slicing for deep climbs (`>= 500 m`).
3. **Independent glide wind (`truth_glide_wind`):** Huber regression of multi-heading glide groundspeeds controlled by glider class (`CCC`, `EN-D`, `EN-C`) as a zero-recentering-bias cross-check on thermal wind.
4. **Multi-pilot ASL top-of-lift (`truth_ceiling`):** calibrated metres ASL (`>= 3` pilots, `>= 5` sustained thermals) with an airspace-cap flag and the real thermal centroid (`centroid_lat`, `centroid_lon`) for fair verification of 1–2.5 km regional weather models.
5. **Core climb rate (`truth_climb_groups`) & task truth (`truth_daily`, `truth_tasks`):** peak sustained 60 s and 30 s core climb rates (`rate_core60_ms`, `rate_core30_ms`) separated from weak searching turns, plus official task outcomes joined with IGC-measured conditions.
6. **Quantitative 1 Hz vs 5 s audit (`audit_1hz_vs_5s.json`):** compares raw 1 Hz IGC tracks (GPS + barometric) against 5 s `fixes/` on identical thermals to verify empirically whether 5 s is sensor-grade.

Tasks are cached in batches under `database/_truth_parts/`, so if Colab disconnects, run again to resume.

## 1. Settings

In [ ]:
DRIVE_FOLDER_ID = "1_JO0XMMQ4NKVfpeA8uKQRkZZPNC4sqZ7"   # project folder in Google Drive (the id at the end of its link)
REPO_URL = "https://github.com/RobertojonesFAI/Pre-task-analyzer.git"
BRANCH = "main"          # set to "feat/igc-truth-sensor" if running before merge
REPO = "/content/Pre-task-analyzer"

WORKERS = 4              # parallel worker processes (4–8 on High-RAM Colab)
CHECK_1HZ_N = 300        # stratified sample of raw 1 Hz flights to audit against 5 s fixes (0 to skip)
LIMIT = None             # set to e.g. 50 for a fast smoke test; None runs every task
REDO = False             # True discards cached batches in database/_truth_parts

Review the settings above before mounting Drive:
- For a first smoke test, set `LIMIT = 50` and `CHECK_1HZ_N = 50` (~1 min).
- For the full database run, keep `LIMIT = None` and `CHECK_1HZ_N = 300` (~5–10 min on a 4–8 CPU Colab runtime).

## 2. Mount Drive, get the code

In [ ]:
import json, os, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import auth, drive
from googleapiclient.discovery import build

auth.authenticate_user()
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")
service = build("drive", "v3")
FIELDS = "id, name, parents, driveId, ownedByMe"


def path_up(file_id, root):
    """Folder names from a file up to the top of its drive, joined under root."""
    names, current = [], file_id
    while True:
        meta = service.files().get(fileId=current, fields=FIELDS, supportsAllDrives=True).execute()
        if not meta.get("parents"):
            return os.path.join(root, *reversed(names))
        names.append(meta["name"])
        current = meta["parents"][0]


def drive_path(folder_id):
    """A writable local path for a Drive folder, from its id."""
    meta = service.files().get(fileId=folder_id, fields=FIELDS, supportsAllDrives=True).execute()
    if meta.get("driveId"):
        drive_name = service.drives().get(driveId=meta["driveId"]).execute()["name"]
        return path_up(folder_id, f"/content/drive/Shareddrives/{drive_name}")
    if meta.get("ownedByMe"):
        return path_up(folder_id, "/content/drive/MyDrive")
    page = None
    while True:
        res = service.files().list(
            q="mimeType='application/vnd.google-apps.shortcut' and trashed=false",
            fields="nextPageToken, files(id, shortcutDetails)", pageToken=page).execute()
        for f in res.get("files", []):
            if f.get("shortcutDetails", {}).get("targetId") == folder_id:
                return path_up(f["id"], "/content/drive/MyDrive")
        page = res.get("nextPageToken")
        if not page:
            raise SystemExit(f"'{meta['name']}' is shared with you. Add a shortcut to it in your Drive "
                             "(right click -> Organize -> Add shortcut), then rerun this cell.")


ROOT = drive_path(DRIVE_FOLDER_ID)
DB = os.path.join(ROOT, "database")
DEM_DIR = os.path.join(ROOT, "dem", "glo30")
OUT = os.path.join(DB, "truth")
assert os.path.isfile(os.path.join(DB, "thermals.parquet")), "Run notebook 08 first: database/thermals.parquet is missing"
subprocess.run(["pip", "install", "-q", "rasterio"], check=True)
print("Database in", DB)

if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "fetch", "-q", "origin"], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", "-q", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "pull", "-q", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, REPO], check=True)


def run(*cmd):
    """Run a script and stream its output; fails fast on non-zero exit."""
    proc = subprocess.Popen(["python3", "-u", *cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in proc.stdout:
            print(line, end="")
        code = proc.wait()
    except KeyboardInterrupt:
        proc.terminate()
        proc.wait()
        print("Stopped.")
        raise
    if code != 0:
        raise RuntimeError(f"{os.path.basename(cmd[0])} stopped with exit code {code}")
    return code

Drive is mounted, `rasterio` is installed for the single-pass takeoff DEM check, and `BRANCH` is synced.

## 3. Build the IGC atmospheric truth tables

Runs `scripts/build_igc_truth.py`: audits takeoff+landing altitude calibration, extracts hodograph + integer-turn wind and 60 s/30 s core climb rates from `fixes/`, pools wind, glide-wind, ceiling, climb, daily and task tables, and executes the `--check-1hz` raw vs 5 s audit.

In [ ]:
cmd = [
    f"{REPO}/scripts/build_igc_truth.py",
    DB,
    "--dem", DEM_DIR,
    "--local", "/content/dem_glo30",
    "--root", ROOT,
    "--workers", str(WORKERS),
    "--check-1hz", str(CHECK_1HZ_N),
]
if LIMIT is not None:
    cmd += ["--limit", str(LIMIT)]
if REDO:
    cmd.append("--redo")

run(*cmd)

Check the printed table counts, free+middle wind summary vs ERA5, and the 1 Hz vs 5 s verdict before examining the detailed diagnostics below.

## 4. Flight calibration & quality audit (`truth_flights`)

Checks how many flights pass dual-DEM calibration (`gold`: `step <= 2 s` and `|takeoff_agl_raw − landing_agl_raw| <= 40 m`; `silver`: landing-calibrated `step <= 5 s`; `excluded`: uncalibrated or coarse live-tracking).

In [ ]:
fl = pd.read_parquet(os.path.join(OUT, "truth_flights.parquet"))
summary = json.loads(open(os.path.join(OUT, "summary.json"), encoding="utf-8").read())

tier_tbl = fl.groupby("cal_tier").agg(
    flights=("track_id", "size"),
    median_step_s=("step_s", "median"),
    median_abs_offset_m=("agl_offset", lambda s: float(s.abs().median())),
    median_abs_drift_m=("alt_consistency_m", lambda s: float(s.abs().median())),
)
tier_tbl["share"] = (tier_tbl["flights"] / len(fl)).round(3)
display(tier_tbl)

display(pd.crosstab(fl["source"], fl["cal_tier"], normalize="index").round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
ax = axes[0]
d_ok = fl["alt_consistency_m"].dropna()
d_clip = d_ok[d_ok.between(-150, 150)]
ax.hist(d_clip, bins=60, color="#2a78d6", alpha=0.85)
ax.axvline(-40, color="#cf5222", ls="--", lw=1.2, label="Gold threshold (±40 m)")
ax.axvline(40, color="#cf5222", ls="--", lw=1.2)
ax.set_xlabel("takeoff_agl_raw − landing_agl_raw (m)")
ax.set_ylabel("Flights")
ax.set_title("Dual-DEM altitude consistency across flights")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)

ax = axes[1]
fl["year"] = pd.to_datetime(fl["flight_date"]).dt.year
yr = pd.crosstab(fl["year"], fl["cal_tier"])[[c for c in ["gold", "silver", "excluded"] if c in fl["cal_tier"].unique()]]
yr.plot(kind="bar", stacked=True, ax=ax, color=["#2c9f74", "#2a78d6", "#c5c3ba"])
ax.set_title("Calibration tiers by year")
ax.set_ylabel("Flights")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

Confirm that `gold` + `silver` retain the bulk of competition flights while cleanly excluding uncalibrated tracks and barometric-drift outliers.

## 5. Quantitative 1 Hz raw IGC vs 5 s fixes audit

Evaluates whether downsampling to 5 s in `database/fixes/` introduces any material error on clean wind, peak 60 s / 30 s core climb rates (including against 1 Hz barometric altitude), or thermal top altitude.

In [ ]:
audit = json.loads(open(os.path.join(OUT, "audit_1hz_vs_5s.json"), encoding="utf-8").read())
print(json.dumps(audit, indent=2))

audit_pq = os.path.join(OUT, "audit_1hz_vs_5s.parquet")
if os.path.isfile(audit_pq):
    ad = pd.read_parquet(audit_pq)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

    ax = axes[0]
    ok = ad["u_clean_1hz"].notna() & ad["u_clean_5s"].notna()
    if ok.any():
        s1 = np.hypot(ad.loc[ok, "u_clean_1hz"], ad.loc[ok, "v_clean_1hz"])
        s5 = np.hypot(ad.loc[ok, "u_clean_5s"], ad.loc[ok, "v_clean_5s"])
        ax.scatter(s1, s5, s=8, alpha=0.35, color="#2a78d6")
        lim = max(float(s1.quantile(0.99)), float(s5.quantile(0.99)), 5.0)
        ax.plot([0, lim], [0, lim], "k--", lw=1)
        ax.set_xlabel("1 Hz clean wind speed (m/s)")
        ax.set_ylabel("5 s clean wind speed (m/s)")
        ax.set_title(f"Thermal wind: 1 Hz vs 5 s (n={ok.sum():,})")
    ax.spines[["top", "right"]].set_visible(False)

    ax = axes[1]
    ok = ad["core60_1hz"].notna() & ad["core60_5s"].notna()
    if ok.any():
        ax.scatter(ad.loc[ok, "core60_1hz"], ad.loc[ok, "core60_5s"], s=8, alpha=0.35, color="#2c9f74")
        lim = max(float(ad.loc[ok, "core60_1hz"].quantile(0.99)), 4.0)
        ax.plot([0, lim], [0, lim], "k--", lw=1)
        ax.set_xlabel("1 Hz peak 60 s climb (m/s)")
        ax.set_ylabel("5 s peak 60 s climb (m/s)")
        ax.set_title(f"Core 60 s climb: 1 Hz vs 5 s (n={ok.sum():,})")
    ax.spines[["top", "right"]].set_visible(False)

    ax = axes[2]
    ok = ad["core60_baro_1hz"].notna() & ad["core60_5s"].notna()
    if ok.any():
        ax.scatter(ad.loc[ok, "core60_baro_1hz"], ad.loc[ok, "core60_5s"], s=8, alpha=0.35, color="#cf5222")
        lim = max(float(ad.loc[ok, "core60_baro_1hz"].quantile(0.99)), 4.0)
        ax.plot([0, lim], [0, lim], "k--", lw=1)
        ax.set_xlabel("1 Hz barometric peak 60 s climb (m/s)")
        ax.set_ylabel("5 s GPS peak 60 s climb (m/s)")
        ax.set_title(f"Baro 1 Hz vs GPS 5 s (n={ok.sum():,})")
    else:
        ax.text(0.5, 0.5, "No baro tracks in sample", ha="center", va="center")
    ax.spines[["top", "right"]].set_visible(False)

    plt.tight_layout()
    plt.show()

If `verdict == "5s_sufficient"`, we have empirical proof that re-parsing the 153k raw IGC archives is unnecessary for atmospheric verification.

## 6. Clean wind (hodograph + integer-turn) vs old wind vs glide wind vs ERA5

Checks whether the new QC-gated thermal wind reduces vector error and shrinks the `−0.5 m/s` (and `−1.1 m/s` late-morning) speed bias against ERA5 and against independent multi-heading glide wind (`truth_glide_wind`).

In [ ]:
wg = pd.read_parquet(os.path.join(OUT, "truth_wind_groups.parquet"))
gw = pd.read_parquet(os.path.join(OUT, "truth_glide_wind.parquet"))

rows = []
for band, methods in (summary.get("wind_vs_era5") or {}).items():
    for mname, met in methods.items():
        if met.get("groups"):
            rows.append({"band": band, "method": mname, **met})
if rows:
    display(pd.DataFrame(rows))

if summary.get("thermal_vs_glide_wind"):
    print("\nIndependent cross-check against multi-heading glide wind (same chip-hour-band):")
    display(pd.DataFrame([{"pair": k, **v} for k, v in summary["thermal_vs_glide_wind"].items() if v.get("groups")]))

if "era5_u" in wg.columns:
    fm = wg[wg["band"].isin(["free air", "middle"])].dropna(subset=["era5_u", "clean_u", "old_u"]).copy()
    fm["era_spd"] = np.hypot(fm["era5_u"], fm["era5_v"])
    fm["clean_bias"] = np.hypot(fm["clean_u"], fm["clean_v"]) - fm["era_spd"]
    fm["old_bias"] = np.hypot(fm["old_u"], fm["old_v"]) - fm["era_spd"]
    fm["hodo_bias"] = np.hypot(fm["hodo_u"], fm["hodo_v"]) - fm["era_spd"]
    fm["clean_err"] = np.hypot(fm["clean_u"] - fm["era5_u"], fm["clean_v"] - fm["era5_v"])
    fm["old_err"] = np.hypot(fm["old_u"] - fm["era5_u"], fm["old_v"] - fm["era5_v"])

    by_hr = fm[fm["solar_hour"].between(10, 17)].groupby("solar_hour").agg(
        n=("clean_bias", "size"),
        old_bias=("old_bias", "median"),
        clean_bias=("clean_bias", "median"),
        hodo_bias=("hodo_bias", "median"),
        old_rmse=("old_err", lambda s: float(np.sqrt(np.mean(s * s)))),
        clean_rmse=("clean_err", lambda s: float(np.sqrt(np.mean(s * s)))),
    ).reset_index()

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
    ax = axes[0]
    ax.plot(by_hr["solar_hour"], by_hr["old_bias"], "o-", color="#9e9d93", label="Old endpoint wind − ERA5")
    ax.plot(by_hr["solar_hour"], by_hr["clean_bias"], "o-", color="#2c9f74", label="Clean (hodo+int) − ERA5")
    ax.plot(by_hr["solar_hour"], by_hr["hodo_bias"], "s--", color="#2a78d6", alpha=0.8, label="Hodograph only − ERA5")
    ax.axhline(0, color="#444", lw=0.9, ls=":")
    ax.set_xlabel("Local solar hour")
    ax.set_ylabel("Median speed bias vs ERA5 (m/s)")
    ax.set_title("Diurnal wind speed bias (free air + middle)")
    ax.legend(frameon=False, fontsize=8.5)
    ax.spines[["top", "right"]].set_visible(False)

    ax = axes[1]
    ax.plot(by_hr["solar_hour"], by_hr["old_rmse"], "o-", color="#9e9d93", label="Old endpoint wind RMSE")
    ax.plot(by_hr["solar_hour"], by_hr["clean_rmse"], "o-", color="#2c9f74", label="Clean (hodo+int) RMSE")
    ax.set_xlabel("Local solar hour")
    ax.set_ylabel("Vector RMSE vs ERA5 (m/s)")
    ax.set_title("Vector RMSE vs ERA5 by solar hour")
    ax.legend(frameon=False, fontsize=8.5)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.show()

Examine both the reduction in vector RMSE and whether the `clean_vs_glide` agreement is tighter than `old_vs_glide`.

## 7. Top of lift (ASL multi-pilot) & Climbing speed (`core60`) diagnostics

In [ ]:
ceil = pd.read_parquet(os.path.join(OUT, "truth_ceiling.parquet"))
climb = pd.read_parquet(os.path.join(OUT, "truth_climb_groups.parquet"))
daily = pd.read_parquet(os.path.join(OUT, "truth_daily.parquet"))

print("Ceiling summary vs ERA5:", json.dumps(summary.get("ceiling_vs_era5", {}), indent=2))

# Day-to-day within-site correlation on sites with >= 5 days
if {"day_ceiling_p90_asl", "day_era5_blh_max_asl"} <= set(daily.columns):
    d_sub = daily.dropna(subset=["day_ceiling_p90_asl", "day_era5_blh_max_asl"])
    within = d_sub.groupby("site_id").filter(lambda x: len(x) >= 5)
    per_site = within.groupby("site_id").apply(lambda x: x["day_ceiling_p90_asl"].corr(x["day_era5_blh_max_asl"]), include_groups=False)
    print(f"Day-to-day ASL ceiling vs ERA5 BLH ASL across {per_site.notna().sum()} sites (>=5 days): median r = {per_site.median():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
ax = axes[0]
if "era5_blh_asl" in ceil.columns:
    c_ok = ceil[~ceil["airspace_cap_flag"]].dropna(subset=["ceiling_p90_asl", "era5_blh_asl"])
    c_cap = ceil[ceil["airspace_cap_flag"]].dropna(subset=["ceiling_p90_asl", "era5_blh_asl"])
    ax.scatter(c_ok["era5_blh_asl"], c_ok["ceiling_p90_asl"], s=6, alpha=0.2, color="#2a78d6", label=f"Normal ({len(c_ok):,})")
    if len(c_cap):
        ax.scatter(c_cap["era5_blh_asl"], c_cap["ceiling_p90_asl"], s=12, alpha=0.65, color="#cf5222", label=f"Airspace cap flag ({len(c_cap):,})")
    lim = max(float(c_ok["era5_blh_asl"].quantile(0.99)), float(c_ok["ceiling_p90_asl"].quantile(0.99)), 4000.0) if len(c_ok) else 4500.0
    ax.plot([0, lim], [0, lim], "k--", lw=1, label="1:1 line")
    ax.set_xlabel("ERA5 BLH top (m ASL)")
    ax.set_ylabel("Calibrated multi-pilot p90 ceiling (m ASL)")
    ax.set_title("Top of lift ASL vs ERA5 BLH ASL")
    ax.legend(frameon=False, fontsize=8.5)
ax.spines[["top", "right"]].set_visible(False)

ax = axes[1]
cl_hr = climb[climb["solar_hour"].between(10, 17)].groupby("solar_hour").agg(
    core60_p75=("core60_p75_ms", "median"),
    core60_p50=("core60_p50_ms", "median"),
    mean_p50=("mean_rate_p50_ms", "median"),
).reset_index()
ax.plot(cl_hr["solar_hour"], cl_hr["core60_p75"], "o-", color="#cf5222", label="Core 60 s (p75)")
ax.plot(cl_hr["solar_hour"], cl_hr["core60_p50"], "o-", color="#e07b39", label="Core 60 s (p50)")
ax.plot(cl_hr["solar_hour"], cl_hr["mean_p50"], "s--", color="#2a78d6", label="Full-thermal mean (p50)")
ax.set_xlabel("Local solar hour")
ax.set_ylabel("Climb rate (m/s)")
ax.set_title("Diurnal climb strength: core 60 s vs full-thermal mean")
ax.legend(frameon=False, fontsize=8.5)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 8. IGC physical truth vs official competition task outcomes (`truth_tasks`)

Verifies how strongly IGC-measured thermal climb strength (`task_core60_p75_ms`) and route-projected tailwind (`task_tailwind_measured_ms`) explain the official `winner_kmh`.

In [ ]:
tt = pd.read_parquet(os.path.join(OUT, "truth_tasks.parquet"))
t_ok = tt[(~tt["stopped"].fillna(False).astype(bool)) & tt["winner_kmh"].notna()].copy()
print(f"Tasks in truth_tasks: {len(tt):,} ({len(t_ok):,} completed with winner_kmh)")
for col in ("task_core60_p75_ms", "task_mean_rate_p50_ms", "task_tailwind_measured_ms", "task_ceiling_p90_asl"):
    if col in t_ok.columns:
        sub = t_ok.dropna(subset=[col, "winner_kmh"])
        if len(sub) > 10:
            print(f"  corr(winner_kmh, {col:26s}) = {sub['winner_kmh'].corr(sub[col]):+.3f}  (n={len(sub):,})")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
ax = axes[0]
s1 = t_ok.dropna(subset=["task_core60_p75_ms", "winner_kmh"])
ax.scatter(s1["task_core60_p75_ms"], s1["winner_kmh"], s=10, alpha=0.35, color="#cf5222")
ax.set_xlabel("IGC measured core 60 s climb p75 (m/s)")
ax.set_ylabel("Official winner speed (km/h)")
ax.set_title("Winner speed vs measured thermal strength")
ax.spines[["top", "right"]].set_visible(False)

ax = axes[1]
if "task_tailwind_measured_ms" in t_ok.columns:
    s2 = t_ok.dropna(subset=["task_tailwind_measured_ms", "winner_kmh"])
    ax.scatter(s2["task_tailwind_measured_ms"], s2["winner_kmh"], s=10, alpha=0.35, color="#2a78d6")
    ax.axvline(0, color="#666", ls=":", lw=1)
    ax.set_xlabel("IGC measured tailwind along route (m/s)")
    ax.set_ylabel("Official winner speed (km/h)")
    ax.set_title("Winner speed vs measured route tailwind")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

### Next step

Once `database/truth/` is generated and verified here, proceed to **Phase 2 (`notebooks/16_forecast_multivar_eval.ipynb`)**: extend the Open-Meteo historical forecast cache with thermodynamic variables (without re-downloading wind) and score every forecast model against **clean wind**, **ASL top-of-lift**, **core climb speed**, and **end-to-end task speed**, deriving leakage-free ERA5 bias correction and composite model weights.